In [1]:
from pathlib import Path
import json
import pandas as pd

In [2]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

COUNTRY_FILES = {
    "US": DATA_DIR / "USvideos.csv",
    "GB": DATA_DIR / "GBvideos.csv",
    "CA": DATA_DIR / "CAvideos.csv",
    "IN": DATA_DIR / "INvideos.csv",
}

frames = []

for country, path in COUNTRY_FILES.items():
    if path.exists():
        part = pd.read_csv(path)
        part["country"] = country
        frames.append(part)

if not frames:
    raise FileNotFoundError(
        "No country CSVs found in data/. Copy your source CSVs here first."
    )

raw = pd.concat(frames, ignore_index=True)

# Parse the original yy.dd.mm date format.
raw["trending_date"] = pd.to_datetime(
    raw["trending_date"].astype(str),
    format="%y.%d.%m",
    errors="coerce",
)

# Standardize column names and types.
raw["video_id"] = raw["video_id"].astype("string").str.strip()
raw["category_id"] = raw["category_id"].astype("string").str.strip()

for col in ["views", "likes", "dislikes", "comment_count"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

# Use a stable video-level lookup: one row per video_id.
video_columns = [
    "video_id", "title", "channel_title", "category_id",
    "publish_time", "tags", "thumbnail_link", "description",
]

video_columns = [c for c in video_columns if c in raw.columns]

videos = (
    raw.dropna(subset=["video_id"])
    .drop_duplicates(subset=["video_id"], keep="first")
    [video_columns]
    .copy()
)

# A record represents one country's observation of a video on a date.
record_columns = [
    "video_id", "country", "trending_date",
    "views", "likes", "dislikes", "comment_count",
]

record_columns = [c for c in record_columns if c in raw.columns]

trending_records = (
    raw.dropna(subset=["video_id", "country", "trending_date"])
    [record_columns]
    .copy()
    .reset_index(drop=True)
)

trending_records.insert(
    0, "record_id", range(1, len(trending_records) + 1)
)

# Load categories from categories.csv if already available.
category_csv = DATA_DIR / "categories.csv"
category_json = DATA_DIR / "US_category_id.json"

if category_csv.exists():
    categories = pd.read_csv(category_csv)

elif category_json.exists():
    with open(category_json, "r", encoding="utf-8") as f:
        category_data = json.load(f)

    categories = pd.DataFrame([
        {
            "category_id": item["id"],
            "category_name": item["snippet"]["title"],
        }
        for item in category_data.get("items", [])
    ])

    categories.to_csv(category_csv, index=False)

else:
    raise FileNotFoundError(
        "Provide categories.csv or US_category_id.json in data/."
    )

# Normalize the category lookup to two columns.
categories = categories[["category_id", "category_name"]].copy()
categories["category_id"] = categories["category_id"].astype("string")
categories = categories.drop_duplicates("category_id")

# Keep category IDs as strings consistently.
videos["category_id"] = videos["category_id"].astype("string")

# Export files.
categories.to_csv(DATA_DIR / "categories.csv", index=False)
videos.to_csv(DATA_DIR / "videos.csv", index=False)
trending_records.to_csv(
    DATA_DIR / "trending_records.csv", index=False
)

print("CSV files prepared:")
print("categories:", len(categories))
print("videos:", len(videos))
print("trending_records:", len(trending_records))
print("Unique video IDs:", videos["video_id"].nunique())
print("Duplicate video IDs in videos:", videos["video_id"].duplicated().sum())
print("Duplicate record IDs:", trending_records["record_id"].duplicated().sum())


CSV files prepared:
categories: 31
videos: 14699
trending_records: 46230
Unique video IDs: 14699
Duplicate video IDs in videos: 0
Duplicate record IDs: 0


In [3]:

for filename in [
    "categories.csv", "videos.csv", "trending_records.csv"
]:
    path = DATA_DIR / filename
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"{filename}: {size_mb:.2f} MB")

    if size_mb > 100:
        print(
            f"WARNING: {filename} exceeds 100 MB. "
            "Remove description, tags, and thumbnail_link if present."
        )


categories.csv: 0.00 MB
videos.csv: 3.51 MB
trending_records.csv: 2.56 MB
